In [1]:
# install dependencies
!pip install -q transformers torch sentencepiece accelerate
!pip install -q bitsandbytes
!pip install -q datasets
!pip install -q errant
!pip install -q sacrebleu
!pip install -q gector
!python -m spacy download en_core_web_sm -q
!python -m spacy download es_core_news_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 151.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 130.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from datasets import load_dataset

# load both datasets
bea19_ds = load_dataset("juancavallotti/bea-19-fine-tune")
multi_ds = load_dataset("juancavallotti/multilingual-gec")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
import random
from collections import Counter

def pair_split(hf_dataset):
    """Keep all languages. sentence=correct, modified=incorrect."""
    pairs = []
    for row in hf_dataset:
        target  = str(row["sentence"]).strip()
        src  = str(row["modified"]).strip()
        lang = str(row.get("lang", "en")).strip()

        if src.lower().startswith("fix grammar:"):
            src = src[len("fix grammar:"):].strip()

        if src and target and src != target:
            pairs.append({"source": src, "target": target, "lang": lang})
    return pairs

# add the pairs from both datasets
bea19_pairs = []
for split_name in bea19_ds.keys():
    bea19_pairs += pair_split(bea19_ds[split_name])
print(f"BEA-19 total pairs: {len(bea19_pairs)}")

multi_pairs = []
for split_name in multi_ds.keys():
    multi_pairs += pair_split(multi_ds[split_name])
print(f"Multilingual total pairs: {len(multi_pairs)}")

# combine the two datset pairs and then do test/train split
all_pairs = bea19_pairs + multi_pairs
print(f"\nTotal combined pairs: {len(all_pairs)}")

random.seed(42)
random.shuffle(all_pairs)

split = int(len(all_pairs) * 0.75)
train_data = all_pairs[:split]
test_data  = all_pairs[split:]
print(f"\nTrain: {len(train_data):,}  |  Test: {len(test_data):,}")

BEA-19 total pairs: 25550
Multilingual total pairs: 217609

Total combined pairs: 243159

Train: 182,369  |  Test: 60,790


In [4]:
EVAL_SIZE = 200
random.seed(42)

# split test set by language
test_en = [p for p in test_data if p["lang"] == "en"]
test_es = [p for p in test_data if p["lang"] == "es"]

print(f"Test set — English: {len(test_en):,}  |  Spanish: {len(test_es):,}")

# sample some for eval set
eval_en = random.sample(test_en, min(EVAL_SIZE, len(test_en)))
eval_es = random.sample(test_es, min(EVAL_SIZE, len(test_es)))

sources_en = [d['source'] for d in eval_en]
references_en = [d['target'] for d in eval_en]

sources_es = [d['source'] for d in eval_es]
references_es = [d['target'] for d in eval_es]

# few-shot examples must come from training data only
few_shot_examples = [ex for ex in train_data if ex["lang"] == "en" and 5 < len(ex["source"].split()) < 25][:3]

print(f"\nEval sizes — English: {len(eval_en)}  |  Spanish: {len(eval_es)}")

Test set — English: 19,185  |  Spanish: 17,137

Eval sizes — English: 200  |  Spanish: 200


In [5]:
import errant
from sacrebleu.metrics import CHRF
from nltk.translate.chrf_score import chrf_precision_recall_fscore_support

annotator  = errant.load('en')
chrf_metric = CHRF()

# english ERRANT score
def errant_score(sources, hypotheses, references):
    tp = fp = fn = 0
    for src, hyp, ref in zip(sources, hypotheses, references):
        try:
            orig = annotator.parse(src)
            hyp_doc = annotator.parse(hyp)
            ref_doc = annotator.parse(ref)
            hyp_edits = {(e.o_start, e.o_end, e.c_str) for e in annotator.annotate(orig, hyp_doc)}
            ref_edits = {(e.o_start, e.o_end, e.c_str) for e in annotator.annotate(orig, ref_doc)}
            tp += len(hyp_edits & ref_edits)
            fp += len(hyp_edits - ref_edits)
            fn += len(ref_edits - hyp_edits)
        except Exception:
            continue

    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f05 = (1.25 * p * r) / (0.25 * p + r) if (p + r) > 0 else 0.0
    return {
        'Precision': round(p,   4),
        'Recall': round(r,   4),
        'F0.5': round(f05, 4),
        'TP': tp,
        'FP': fp,
        'FN': fn
    }

# no ERRANT score for Spanish, so use CHRF
def chrf_score(hypotheses, references):
    precisions = []
    recalls = []
    f2_scores = []

    for hyp, ref in zip(hypotheses, references):
        p, r, f, _ = chrf_precision_recall_fscore_support(ref, hyp, 6, beta=2.0)

        precisions.append(p * 100)
        recalls.append(r * 100)
        f2_scores.append(f * 100)

    return {
        'Precision': round(sum(precisions) / len(precisions), 4),
        'Recall':    round(sum(recalls) / len(recalls), 4),
        'chrF2':     round(sum(f2_scores) / len(f2_scores), 4),
    }

In [6]:
from huggingface_hub import login

hf_token = ""
login(hf_token)

In [7]:
ZERO_SHOT_SYSTEM_EN = (
    "You are a grammatical error correction assistant. "
    "Correct all grammatical errors in the given sentence. "
    "Output ONLY the corrected sentence — no explanation, no extra text."
)

# same instruction but translated from English
ZERO_SHOT_SYSTEM_ES = (
    "Eres un asistente de corrección gramatical. "
    "Corrige todos los errores gramaticales en la oración dada. "
    "Devuelve ÚNICAMENTE la oración corregida — sin explicaciones ni texto adicional."
)

def build_zero_shot(sentence, lang="en"):
    return f"Sentence: {sentence}\nCorrected:"

def build_few_shot(sentence, lang="en"):
    # use English few-shot examples for English
    # for Spanish just do zero-shot
    if lang == "es":
        return build_zero_shot(sentence, lang)
    examples = "\n\n".join(f"Sentence: {ex['source']}\nCorrected: {ex['target']}" for ex in few_shot_examples)
    return f"{examples}\n\nSentence: {sentence}\nCorrected:"

def clean_output(text, source):
    text = text.strip()
    for prefix in ("Corrected:", "Corrected sentence:", "Answer:", "Corregida:", "Oración corregida:"):
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
    return text if text else source

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# load Mistral in by following documentation

LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

# need this so that there is enough memory in T4
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, token=hf_token)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, quantization_config=bnb_config, device_map="auto", token=hf_token)
llm_model.eval()

llm_model.generation_config.max_length = None

# set up a pipeline for text generation
llm_tokenizer.pad_token_id = llm_tokenizer.eos_token_id
llm_tokenizer.padding_side = "left"
llm_pipe = pipeline("text-generation", model=llm_model, tokenizer=llm_tokenizer, max_new_tokens=128, max_length=None, do_sample=False, temperature=None, top_p=None, pad_token_id=llm_tokenizer.eos_token_id)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample', 'top_p', 'pad_token_id', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [9]:
def format_mistral_prompt(user_message, lang="en"):
    system = ZERO_SHOT_SYSTEM_EN if lang == "en" else ZERO_SHOT_SYSTEM_ES
    return f"<s>[INST] {system}\n\n{user_message} [/INST]"

# batch the prompts together and push it through the piepline
def mistral_correct_batch(sentences, few_shot=False, lang="en"):
    prompts = []
    for src in sentences:
        user_msg = build_few_shot(src, lang) if few_shot else build_zero_shot(src, lang)
        prompt = format_mistral_prompt(user_msg, lang)
        prompts.append(prompt)

    results = []

    outputs = llm_pipe(prompts, batch_size=4)

    for src, prompt, out in zip(sentences, prompts, outputs):
        try:
            reply = out[0]['generated_text'][len(prompt):].strip()
            results.append(clean_output(reply, src))
        except Exception as e:
            print(f"  Error parsing output: {e}")
            results.append(src)

    return results

# English results
print("Mistral — English zero-shot")
mistral_en_zs_preds  = mistral_correct_batch(sources_en, few_shot=False, lang="en")
mistral_en_zs_scores = errant_score(sources_en, mistral_en_zs_preds, references_en)
print("EN Zero-Shot:", mistral_en_zs_scores)

print("\nMistral — English few-shot")
mistral_en_fs_preds  = mistral_correct_batch(sources_en, few_shot=True, lang="en")
mistral_en_fs_scores = errant_score(sources_en, mistral_en_fs_preds, references_en)
print("EN Few-Shot:", mistral_en_fs_scores)

# Spanish results
print("\nMistral — Spanish zero-shot")
mistral_es_zs_preds  = mistral_correct_batch(sources_es, few_shot=False, lang="es")
mistral_es_zs_scores = chrf_score(mistral_es_zs_preds, references_es)
print("ES Zero-Shot:", mistral_es_zs_scores)

print("\nMistral — Spanish few-shot")
mistral_es_fs_preds  = mistral_correct_batch(sources_es, few_shot=True, lang="es")
mistral_es_fs_scores = chrf_score(mistral_es_fs_preds, references_es)
print("ES Few-Shot:", mistral_es_fs_scores)

Mistral — English zero-shot
EN Zero-Shot: {'Precision': 0.2908, 'Recall': 0.4753, 'F0.5': 0.3153, 'TP': 212, 'FP': 517, 'FN': 234}

Mistral — English few-shot
EN Few-Shot: {'Precision': 0.3151, 'Recall': 0.5045, 'F0.5': 0.3407, 'TP': 225, 'FP': 489, 'FN': 221}

Mistral — Spanish zero-shot
ES Zero-Shot: {'Precision': 83.7776, 'Recall': 85.4163, 'chrF2': 84.8291}

Mistral — Spanish few-shot
ES Few-Shot: {'Precision': 83.7776, 'Recall': 85.4163, 'chrF2': 84.8291}
